# Revenue Forecasting

**InsightForge Business Analytics Studio — Python Analyst Toolkit**

## Purpose
Produce a simple, fully transparent short-term revenue forecast: no black-box model, just a moving average for smoothing plus a linear trend extrapolated forward.

## Methodology
1. Aggregate order revenue to **daily** totals.
2. Compute a trailing **moving average** (default: 7-day window) to smooth day-to-day noise and visualize the underlying pattern.
3. Fit an **ordinary least-squares linear trend** (`numpy.polyfit`, degree 1) to the full daily revenue history, using day-offset-from-start as the independent variable.
4. Extrapolate that line **30 days** beyond the last date in the data to produce the forecast.
5. Compute the standard deviation of the linear fit's residuals as a simple, transparent stand-in for uncertainty -- **not** a formal statistical confidence interval (see Limitations).

## Why not a 'real' forecasting model?
This toolkit deliberately uses a moving average + linear trend instead of ARIMA, Prophet, or an ML regressor. The goal is a forecast a business owner can fully audit by eye: draw a line through the data, extend it forward. A black-box model might fit historical data better, but it trades away exactly the transparency this toolkit is built to provide.

## Assumptions
- The recent linear trend continues unchanged over the forecast horizon.
- No explicit seasonality (weekly/monthly cycles) is modeled in the trend line itself -- only smoothed out. If your business has strong seasonality, treat this forecast as a baseline, not a seasonally-aware prediction.
- Day-to-day revenue is aggregated across all channels/regions; no segment-level forecasting is performed here.

## Uncertainty / Confidence Explanation
The shaded band in the forecast chart below is **plus/minus one residual standard deviation** from the historical linear fit -- a rough measure of how noisy actual daily revenue has been around the trend line historically. It is **not** a rigorous prediction interval (it doesn't grow with forecast horizon, doesn't account for trend uncertainty, and assumes residuals are roughly stationary). Treat it as 'this is roughly how much day-to-day revenue has wobbled around the trend historically,' not a statistically calibrated confidence interval.

## Limitations
- **This is not a guarantee of future results.** Real businesses are affected by market conditions, competition, seasonality, promotions, and one-off events that a simple linear trend cannot anticipate.
- Extrapolating a straight line arbitrarily far into the future becomes increasingly unreliable -- this notebook limits itself to a 30-day horizon for that reason.
- A sudden change in trend direction (e.g. a new competitor, a marketing campaign ending) will not be reflected in the forecast until new data is added and the model is re-fit.

In [1]:
import os
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

TOOLKIT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, TOOLKIT_DIR)
from clean_data import load_csv, clean_dataframe
from analysis_utils import moving_average, forecast_linear_trend, residual_std, fit_linear_trend

OUTPUT_DIR = os.path.join(TOOLKIT_DIR, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

MOVING_AVERAGE_WINDOW_DAYS = 7
FORECAST_HORIZON_DAYS = 30

orders_path = os.path.join(TOOLKIT_DIR, "sample_data", "orders.csv")
raw_orders = load_csv(orders_path)
orders, cleaning_summary = clean_dataframe(raw_orders)
print(cleaning_summary)

Data Cleaning Summary

- Removed 11 exact duplicate row(s) (kept first occurrence).
- No invalid negative values found in quantity/revenue/cost/discount/rating.
- Missing values detected (left as-is; not auto-filled to avoid inventing data):
    - 'campaign': 3831 missing (60.6% of rows)
    - 'region': 6 missing (0.1% of rows)
    - 'marketingSource': 4 missing (0.1% of rows)
    - 'discount': 3 missing (0.0% of rows)
    - 'channel': 2 missing (0.0% of rows)
- Statistical outliers detected via IQR method (reported only, not removed):
    - 'quantity': 201 value(s) outside [-0.5, 3.5]
    - 'revenue': 580 value(s) outside [-39.06, 185.38]
    - 'cost': 507 value(s) outside [-20.03, 94.18]
    - 'profit': 442 value(s) outside [-30.52, 105.58]
    - 'discount': 615 value(s) outside [-0.08, 0.12]
  Outliers are left in the cleaned data by default -- in real business data these are often legitimate large orders, not errors. Review manually before excluding.

Rows before cleaning: 6326
Row

In [2]:
required_cols = {"date", "revenue"}
if not required_cols.issubset(orders.columns):
    print(f"Forecasting unavailable: missing column(s) {sorted(required_cols - set(orders.columns))}")
    daily_revenue = None
else:
    daily_revenue = orders.groupby("date")["revenue"].sum().sort_index()
    # Fill any gap days with 0 revenue so the daily series is evenly spaced.
    full_index = pd.date_range(daily_revenue.index.min(), daily_revenue.index.max(), freq="D")
    daily_revenue = daily_revenue.reindex(full_index, fill_value=0.0)
    print(f"Daily revenue series: {len(daily_revenue)} days, {daily_revenue.index.min().date()} to {daily_revenue.index.max().date()}")
    display(daily_revenue.tail())

Daily revenue series: 547 days, 2024-01-01 to 2025-06-30


2025-06-26    1936.61
2025-06-27    1710.94
2025-06-28    8508.54
2025-06-29    2103.59
2025-06-30    2964.94
Freq: D, Name: revenue, dtype: float64

## Moving Average

In [3]:
if daily_revenue is not None:
    ma = moving_average(daily_revenue.values, window=MOVING_AVERAGE_WINDOW_DAYS)
    ma_series = pd.Series(ma, index=daily_revenue.index)

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(daily_revenue.index, daily_revenue.values, alpha=0.35, label="Daily revenue", color="#999")
    ax.plot(ma_series.index, ma_series.values, label=f"{MOVING_AVERAGE_WINDOW_DAYS}-day moving average", color="#2a5599")
    ax.set_title("Daily Revenue with Moving Average")
    ax.set_ylabel("Revenue ($)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "04_daily_revenue_moving_average.png"), dpi=150)
    plt.close(fig)
    print("Saved output/04_daily_revenue_moving_average.png")

Saved output/04_daily_revenue_moving_average.png


## Linear Trend Forecast

In [4]:
if daily_revenue is not None:
    day_offsets = np.arange(len(daily_revenue))
    trend = fit_linear_trend(day_offsets, daily_revenue.values)
    _trend, future_x, forecast_y = forecast_linear_trend(day_offsets, daily_revenue.values, FORECAST_HORIZON_DAYS)
    uncertainty = residual_std(day_offsets, daily_revenue.values, trend)

    print(f"Fitted trend: revenue ~= {trend.intercept:.2f} + {trend.slope:.4f} * day")
    print(f"Residual std (uncertainty band): +/- ${uncertainty:,.2f} per day")
    print(f"Projected total revenue over next {FORECAST_HORIZON_DAYS} days: ${forecast_y.sum():,.2f}")

Fitted trend: revenue ~= 578.00 + 1.8923 * day
Residual std (uncertainty band): +/- $682.13 per day
Projected total revenue over next 30 days: $49,215.36


In [5]:
if daily_revenue is not None:
    future_dates = pd.date_range(daily_revenue.index.max() + pd.Timedelta(days=1), periods=FORECAST_HORIZON_DAYS, freq="D")

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(daily_revenue.index, daily_revenue.values, alpha=0.3, color="#999", label="Actual daily revenue")
    ax.plot(daily_revenue.index, trend.predict(day_offsets), color="#2a5599", label="Fitted trend")
    ax.plot(future_dates, forecast_y, color="#c96a2e", linestyle="--", label=f"{FORECAST_HORIZON_DAYS}-day forecast")
    ax.fill_between(future_dates, forecast_y - uncertainty, forecast_y + uncertainty, color="#c96a2e", alpha=0.15, label="+/- 1 residual std")
    ax.set_title("Revenue Forecast: Linear Trend Extrapolation")
    ax.set_ylabel("Revenue ($)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "04_revenue_forecast.png"), dpi=150)
    plt.close(fig)

    forecast_table = pd.DataFrame({"date": future_dates, "forecastRevenue": forecast_y, "lowerBound": forecast_y - uncertainty, "upperBound": forecast_y + uncertainty})
    forecast_table.to_csv(os.path.join(OUTPUT_DIR, "04_forecast_table.csv"), index=False)
    print("Saved output/04_revenue_forecast.png and output/04_forecast_table.csv")
    display(forecast_table.head())

Saved output/04_revenue_forecast.png and output/04_forecast_table.csv


,date,forecastRevenue,lowerBound,upperBound
0,2025-07-01,1613.074106,930.943081,2295.205131
1,2025-07-02,1614.966378,932.835353,2297.097403
2,2025-07-03,1616.858649,934.727624,2298.989674
3,2025-07-04,1618.750921,936.619896,2300.881946
4,2025-07-05,1620.643193,938.512168,2302.774218


## Summary
This notebook wrote `04_daily_revenue_moving_average.png`, `04_revenue_forecast.png`, and `04_forecast_table.csv` to `output/`.

**Reminder:** re-read the Limitations section above -- this forecast is a transparent trend extrapolation for planning discussions, not a guarantee of future revenue.